In [1]:
# Kill all processess on GPU
!fuser -v /dev/nvidia* -k

                     USER        PID ACCESS COMMAND
/dev/nvidia0:        root      15647 F...m python3
/dev/nvidiactl:      root      15647 F...m python3
/dev/nvidia-uvm:     root      15647 F...m python3


In [2]:
# Check GPU status
!nvidia-smi

Sun Jul 19 00:08:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   64C    P0             31W /   70W |       0MiB /  15360MiB |      3%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Libraries

In [3]:
%%capture
!uv pip uninstall torchao torchaudio torchvision -y
!uv pip install \
    "transformers==4.53.3" \
    "peft==0.17.1" \
    "trl" \
    "accelerate" \
    "bitsandbytes" \
    "wandb"

In [4]:
import os
import torch
from datetime import datetime
from transformers import AutoModelForQuestionAnswering, AutoTokenizer
from peft import PeftModel
from huggingface_hub import snapshot_download
from datasets import load_dataset, Dataset

# Configurations

In [5]:
# Run configuration
SRC_LANG = 'en'
TGT_LANG = 'vi'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Model configuration
MODEL_ID = 'FacebookAI/xlm-roberta-base'
LEGAMEX_LANG_ID = 'alxxtexxr/XLM-R-Base-wikipedia-vi-5K-LegameX-LoRA-v260718210036'
LEGAMEX_LANG_CKPT_STEP = 120
LEGAMEX_TASK_ID = 'alxxtexxr/XLM-R-Base-squad-en-5K-LegameX-LoRA-v260718203339'
LEGAMEX_TASK_CKPT_STEP = 620

ADDITION_TYPE = 'Averaging'
addition_weights = [0.5, 0.5] if ADDITION_TYPE == 'Averaging' else [1.0, 1.0]

model_id_prefix, model_id_suffix = LEGAMEX_TASK_ID.split(SRC_LANG)
data_size_str = model_id_suffix.split('LegameX-LoRA')[0].replace('-', '')
hub_merged_model_id = f"{model_id_prefix}{TGT_LANG}-{data_size_str}-LegameX-LoRA-{ADDITION_TYPE}-v{datetime.now().strftime("%y%m%d%H%M%S")}"
print(f"Hub merged model ID: {hub_merged_model_id}")

Hub merged model ID: alxxtexxr/XLM-R-Base-squad-vi-5K-LegameX-LoRA-Averaging-v260719000824


# Utilities

In [6]:
# LoRA utilities
def download_hf_model(
        repo_id, 
        ckpt_step, 
        max_checkpoints=10_000,
        ckpt_interval=25,
    ):
    local_dir = repo_id.split('/')[-1]
    ignore_checkpoints = None
    
    if ckpt_step is not None:
        ignore_checkpoints = [f'checkpoint-{i}/*' for i in range(0, max_checkpoints, ckpt_interval) if i != ckpt_step]

    snapshot_download(
        repo_id=repo_id,
        local_dir=local_dir,
        ignore_patterns=ignore_checkpoints,
    )

    ckpt_dir = None
    if ckpt_step is not None:
        ckpt_dir = os.path.join(local_dir, f'checkpoint-{ckpt_step}')
    return local_dir, ckpt_dir

# Model

In [7]:
_, legamex_lang_dir = download_hf_model(repo_id=LEGAMEX_LANG_ID, ckpt_step=LEGAMEX_LANG_CKPT_STEP)
_, legamex_task_dir = download_hf_model(repo_id=LEGAMEX_TASK_ID, ckpt_step=LEGAMEX_TASK_CKPT_STEP)
legamex_lang_tfr_dir = f'{legamex_lang_dir}/tfr'
legamex_task_tfr_dir = f'{legamex_task_dir}/tfr'

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 75 files:   0%|          | 0/75 [00:00<?, ?it/s]

Fetching 255 files:   0%|          | 0/255 [00:00<?, ?it/s]

In [8]:
legamex_lang_tfr_fixed_dir = f'{legamex_lang_dir}/tfr_fixed'
legamex_task_tfr_fixed_dir = f'{legamex_task_dir}/tfr_fixed'

!mkdir -p $legamex_lang_tfr_fixed_dir
!mkdir -p $legamex_task_tfr_fixed_dir
!cp -r $legamex_lang_tfr_dir/* $legamex_lang_tfr_fixed_dir
!cp -r $legamex_task_tfr_dir/* $legamex_task_tfr_fixed_dir
!rm $legamex_lang_tfr_fixed_dir/adapter_model.safetensors
!rm $legamex_task_tfr_fixed_dir/adapter_model.safetensors

In [9]:
from safetensors.torch import load_file, save_file

legamex_lang_tfr_state_dict = load_file(f'{legamex_lang_tfr_dir}/adapter_model.safetensors')
legamex_task_tfr_state_dict = load_file(f'{legamex_task_tfr_dir}/adapter_model.safetensors')

def fix_legamex_state_dict(state_dict, prefix='base_model.model.'):
    return {f'{prefix}{k}': v for k, v in state_dict.items()}

legamex_lang_tfr_fixed_state_dict = fix_legamex_state_dict(legamex_lang_tfr_state_dict)
legamex_task_tfr_fixed_state_dict = fix_legamex_state_dict(legamex_task_tfr_state_dict)

save_file(legamex_lang_tfr_fixed_state_dict, f'{legamex_lang_tfr_fixed_dir}/adapter_model.safetensors')
save_file(legamex_task_tfr_fixed_state_dict, f'{legamex_task_tfr_fixed_dir}/adapter_model.safetensors')

In [10]:
# from collections import OrderedDict

# def fix_legamex_state_dict(state_dict):
#     state_dict_fixed = OrderedDict()

#     lora_tfr_A_weight = None
#     lora_tfr_B_weight = None
#     gate_A_weight = None
#     gate_B_weight = None

#     for key, value in state_dict.items():
#         key_fixed = key.replace('__DOT__', '.')
        
#         if 'lora.tfr.A' in key_fixed:
#             lora_tfr_A_weight = value
#         elif 'lora.tfr.B' in key_fixed:
#             lora_tfr_B_weight = value
#         elif 'gate.A' in key_fixed:
#             gate_A_weight = value
#         elif 'gate.B' in key_fixed:
#             gate_B_weight = value
#         elif 'qa_outputs' in key_fixed:
#             state_dict_fixed[key_fixed] = value
#         else:
#             continue
            
#         if lora_tfr_A_weight is not None and gate_A_weight is not None:
#             lora_A_weight = lora_tfr_A_weight * (1.0 - gate_A_weight)
#             key_fixed = key_fixed.replace('gate.', 'lora_')
#             state_dict_fixed[key_fixed] = lora_A_weight
#             lora_tfr_A_weight = None
#             gate_A_weight = None
#         if lora_tfr_B_weight is not None and gate_B_weight is not None:
#             lora_B_weight = lora_tfr_B_weight * (1.0 - gate_B_weight)
#             key_fixed = key_fixed.replace('gate.', 'lora_')
#             state_dict_fixed[key_fixed] = lora_B_weight
#             lora_tfr_B_weight = None
#             gate_B_weight = None
            
#     return state_dict_fixed

# legamex_lang_state_dict_fixed = fix_legamex_state_dict(legamex_lang_state_dict)
# legamex_task_state_dict_fixed = fix_legamex_state_dict(legamex_task_state_dict)

In [11]:
base_model = AutoModelForQuestionAnswering.from_pretrained(MODEL_ID)

lora_model = PeftModel.from_pretrained(base_model, legamex_task_tfr_fixed_dir, adapter_name='task')

Some weights of XLMRobertaForQuestionAnswering were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
lora_model

PeftModelForFeatureExtraction(
  (base_model): LoraModel(
    (model): XLMRobertaForQuestionAnswering(
      (roberta): XLMRobertaModel(
        (embeddings): XLMRobertaEmbeddings(
          (word_embeddings): Embedding(250002, 768, padding_idx=1)
          (position_embeddings): Embedding(514, 768, padding_idx=1)
          (token_type_embeddings): Embedding(1, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): XLMRobertaEncoder(
          (layer): ModuleList(
            (0-11): 12 x XLMRobertaLayer(
              (attention): XLMRobertaAttention(
                (self): XLMRobertaSdpaSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (task): Dropout(p=0.1, inplace=False)
                    )
                    (lor

In [ ]:
!cat $legamex_task_tfr_fixed_dir/adapter_config.json

{
    "task_type": "FEATURE_EXTRACTION",
    "peft_type": "LORA",
    "auto_mapping": null,
    "base_model_name_or_path": "FacebookAI/xlm-roberta-base",
    "revision": null,
    "inference_mode": true,
    "r": 16,
    "target_modules": [
        "intermediate.dense",
        "query",
        "key",
        "value",
        "output.dense"
    ],
    "exclude_modules": null,
    "lora_alpha": 16,
    "lora_dropout": 0.1,
    "fan_in_fan_out": false,
    "bias": "none",
    "use_rslora": false,
    "modules_to_save": null,
    "init_lora_weights": true,
    "layers_to_transform": null,
    "layers_pattern": null,
    "rank_pattern": {},
    "alpha_pattern": {},
    "megatron_config": null,
    "megatron_core": "megatron.core",
    "trainable_token_indices": null,
    "loftq_config": {},
    "eva_config": null,
    "corda_config": null,
    "use_dora": false,
    "use_qalora": false,
    "qalora_group_size": 16,
    "layer_replication": null,
    "lora_bias": false,
    "target_paramete

: 

In [11]:
base_model = AutoModelForQuestionAnswering.from_pretrained(MODEL_ID)

lora_model = PeftModel.from_pretrained(base_model, legamex_task_tfr_fixed_dir, adapter_name='task')
lora_model.load_adapter(legamex_lang_tfr_fixed_dir, adapter_name='lang')

Some weights of XLMRobertaForQuestionAnswering were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


<All keys matched successfully>

In [12]:
lora_model.add_weighted_adapter(
    adapters=['task', 'lang'],
    weights=addition_weights,
    combination_type='linear',
    adapter_name='legamex_addition'
)
lora_model.set_adapter('legamex_addition')

In [13]:
merged_model = lora_model.merge_and_unload()
merged_model = merged_model.to(DEVICE).eval()

print("device:", merged_model.device)

device: cuda:0


In [ ]:
# Upload the merged model to Hugging Face
merged_model.push_to_hub(hub_merged_model_id)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.push_to_hub(hub_merged_model_id)

print(f"Merged model uploaded to: https://huggingface.co/{hub_merged_model_id}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...xgk09wa/model.safetensors:   4%|4         | 47.1MB / 1.11GB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ..._/sentencepiece.bpe.model: 100%|##########| 5.07MB / 5.07MB            

  ...mp2o47p_d_/tokenizer.json: 100%|##########| 17.1MB / 17.1MB            

Merged model uploaded to: https://huggingface.co/alxxtexxr/XLM-R-Base-squad-vi-5K-LegameX-LoRA-Averaging-v260718221730


: 